In [ ]:
# Standard library
import os
import csv
import warnings
from datetime import datetime

# Third-party libraries
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d

# Matplotlib and related tools
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.patheffects as pe
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Plot styling
import scienceplots
plt.style.use(['science', 'nature'])

# Warnings configuration
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
# preset values for the analysis: (you need to update)
#____________________________________________________________________________________________________________________
# physical constants
Z = 20
A = 40
mass_nucleon = 0.938273
mass_nucleus = A * 0.931494
alpha_fine = 1 / 137.036
Ex_cut = 0.03
Ex_cut_lowq = 0.1

# three-momentum bin centers
qvcenters = [0.100, 0.148, 0.167, 0.205, 0.240, 0.300, 0.380, 0.475, 0.570, 0.649, 0.756, 0.991, 1.619, 1.921, 2.213, 2.500, 2.783, 3.500]
# three-momentum edges
qvbins = [0.063, 0.124, 0.158, 0.186, 0.223, 0.270, 0.340, 0.428, 0.523, 0.609, 0.702, 0.878, 1.302, 1.770, 2.067, 2.357, 2.642, 2.923, 4.500]
# four-momentum squared bin names, in string format
qvbin_names = ['[0.063,0.124]', '[0.124,0.158]', '[0.158,0.186]', '[0.186,0.223]', '[0.223,0.270]', '[0.270,0.340]', '[0.340,0.428]', '[0.428,0.523]', '[0.523,0.609]',
                '[0.609,0.702]', '[0.702,0.878]', '[0.878,1.302]', '[1.302,1.770]', '[1.770,2.067]', '[2.067,2.357]', '[2.357,2.642]', '[2.642,2.923]', '[2.923,4.500]']

# four-momentum squared bin centers
Q2centers = [0.010, 0.020, 0.026, 0.040, 0.056, 0.093, 0.120, 0.160, 0.265, 0.380, 0.500, 0.800, 1.250, 1.750, 2.250, 2.750, 3.250, 3.750]
# four-momentum squared bin edges
Q2bins = [0.004, 0.015, 0.025, 0.035, 0.045, 0.070, 0.100, 0.145, 0.206, 0.322, 0.438, 0.650, 1.050, 1.500, 2.000, 2.500, 3.000, 3.500, 4.000]
# four-momentum squared bin names, in string format
Q2bin_names = ['[0.004,0.015]', '[0.015,0.025]', '[0.025,0.035]', '[0.035,0.045]', '[0.045,0.070]', '[0.070,0.100]', '[0.100,0.145]', '[0.145,0.206]', '[0.206,0.322]',
                '[0.322,0.438]', '[0.438,0.650]', '[0.650,1.050]', '[1.050,1.500]', '[1.500,2.000]', '[2.000,2.500]', '[2.500,3.000]', '[3.000,3.500]', '[3.500,4.000]']

In [ ]:
RLRT_number_file = 'Output/Ca40_SuSAV2_Diff.csv'
with open(RLRT_number_file, 'a', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames = pd.Series(index = ['qvcenter', 'Q2center', 'nu', 'Diff_RL', 'RLerr', 'Diff_RT', 'RTerr']).index)
    writer.writeheader()

In [ ]:
errorbar_setting = {'markersize':0,'capsize':0,'lw':1,'fmt':'D','elinewidth':1,'zorder':1,'alpha':0.5}
our_scatter_setting = {'s':10,'marker':'D','edgecolors':'black','linewidth':0.7,'zorder':3,'alpha':1}
other_scatter_setting = {'s':10,'marker':'D','edgecolors':'black','linewidth':0.7,'zorder':2,'alpha':1}
photo_scatter_setting = {'s':30,'color':'lime','marker':'^','edgecolors':'black','linewidth':0.5,'zorder':2}

def RLRT_plot_qvbin(bin_indices = [0]):
    RL_plot_top = np.array([300, 300, 300, 300, 250, 
                            150, 80, 60, 40, 30, 
                            50, 50, 50, 50, 50, 
                            50, 50, 50])
    RT_plot_top = np.array([50, 50, 50, 60, 80, 
                            120, 120, 120, 120, 120, 
                            100, 100, 50, 50, 50, 
                            50, 50, 50])

    fig, axs = plt.subplots(nrows = len(bin_indices)*2, ncols = 2, figsize = (10, 8), dpi = 300)
    
    handles = []
    labels = []
    
    for i in range(len(bin_indices)):
        bin_index = bin_indices[i]
        qvcenter = qvcenters[bin_index]
        
        # Normal part
        SuSAV2_data = pd.read_csv(f'Data/Ca40_SuSAV2/qv/40Ca_BC_q_{int(qvcenter*1e3):04d}_new.dat', sep = '\s+', header = None)
        SuSAV2_data.columns = ['nu', 'RLQE', 'RTQE', 'RLMEC', 'RTMEC', 'RLinelastic', 'RTinelastic']
        SuSAV2_data['RLtotal'] = SuSAV2_data['RLQE'] + SuSAV2_data['RLMEC'] + SuSAV2_data['RLinelastic']
        SuSAV2_data['RTtotal'] = SuSAV2_data['RTQE'] + SuSAV2_data['RTMEC'] + SuSAV2_data['RTinelastic']
        SuSAV2_data_slice = SuSAV2_data[SuSAV2_data['nu'] > 0].copy()
        line1, = axs[i*2, 0].plot(SuSAV2_data_slice['nu'], SuSAV2_data_slice['RLQE'], label = '$R_L$, $R_T$ (QE) SuSAv2', color = 'violet', linestyle = ':', lw = 3, zorder = -1)
        axs[i*2, 0].plot(SuSAV2_data_slice['nu'], SuSAV2_data_slice['RLtotal'], label = '$R_L$, $R_T$ (Total) SuSAv2', color = 'violet', linestyle = '-', lw = 2, zorder = -1)
        line2, = axs[i*2, 1].plot(SuSAV2_data_slice['nu'], SuSAV2_data_slice['RTQE'], color = 'violet', linestyle = ':', lw = 3, zorder = -1)
        axs[i*2, 1].plot(SuSAV2_data_slice['nu'], SuSAV2_data_slice['RTQE'] + SuSAV2_data_slice['RTMEC'], label = '$R_T$ (QE+2p2h) SuSAv2', color = 'violet', linestyle = '--', lw = 2, zorder = -1)
        axs[i*2, 1].plot(SuSAV2_data_slice['nu'], SuSAV2_data_slice['RTtotal'], color = 'violet', linestyle='-', lw = 2, zorder = -1)
        line1.set_path_effects([pe.Stroke(linewidth = 3.5, foreground = 'black'), pe.Normal()])
        line2.set_path_effects([pe.Stroke(linewidth = 3.5, foreground = 'black'), pe.Normal()])
        
        before = pd.read_csv("Data/Ca40_RLRT_Numbers_Before.csv")
        after = pd.read_csv("Data/Ca40_RLRT_Numbers_After.csv")
        before = before[before['qvcenter'] == qvcenter]
        after = after[after['qvcenter'] == qvcenter]
        before['RL'] = before['RL']*1e3
        before['RLerr'] = before['RLerr']*1e3
        before['RT'] = before['RT']*1e3
        before['RTerr'] = before['RTerr']*1e3
        after['RL'] = after['RL']*1e3
        after['RLerr'] = after['RLerr']*1e3
        after['RT'] = after['RT']*1e3
        after['RTerr'] = after['RTerr']*1e3
        axs[i*2, 0].scatter((before['nu'] + after['nu']) / 2, (before['RL'] + after['RL']) / 2, color = 'red', label = '$R_L$, $R_T$ this analysis', **our_scatter_setting)
        axs[i*2, 0].errorbar((before['nu'] + after['nu']) / 2, (before['RL'] + after['RL']) / 2, yerr = np.sqrt((before['RLerr']**2 + after['RLerr']**2) / 4 + (before['RL'] - after['RL'])**2 / 4), color = 'red', **errorbar_setting)
        axs[i*2, 1].scatter((before['nu'] + after['nu']) / 2, (before['RT'] + after['RT']) / 2, color = 'red', **our_scatter_setting)
        axs[i*2, 1].errorbar((before['nu'] + after['nu']) / 2, (before['RT'] + after['RT']) / 2, yerr = np.sqrt((before['RTerr']**2 + after['RTerr']**2) / 4 + (before['RT'] - after['RT'])**2 / 4), color = 'red', **errorbar_setting)
        
        Photo_data = pd.read_csv('Data/Ca40_Photoproduction_qv.csv')
        Photo_data = Photo_data[Photo_data["qvcenter"] == qvcenter]
        Photo_data['RT'] *= 1e3
        Photo_data['RTerr'] *= 1e3
        axs[i*2, 1].scatter(Photo_data['qvcenter'], Photo_data['RT'], label = '$R_T$ Photo-production ($Q^2=0 \ GeV^2$)', **photo_scatter_setting)
        axs[i*2, 1].errorbar(Photo_data['qvcenter'], Photo_data['RT'], yerr = Photo_data['RTerr'], **errorbar_setting)
        Photo_max = Photo_data['RT'].max() * 1.1
        
        W_peaks = np.array([0.93, 1.07, 1.23]) 
        W_colors = ['darkorange', 'turquoise', 'slateblue']
        W_locations = - mass_nucleon + np.sqrt(qvcenter**2 + W_peaks**2)
        for k in range(len(W_peaks)):
            location = W_locations[k]
            if location < qvcenter:
                axs[i*2, 0].axvline(x = location, color = W_colors[k], linestyle = 'dashdot', lw = 1, label = f'$W={W_peaks[k]}$ $GeV$')
                axs[i*2, 1].axvline(x = location, color = W_colors[k], linestyle = 'dashdot', lw = 1)
        
        axs[i*2, 0].axvline(x = qvcenter, color = 'brown', linestyle='dashdot', lw = 1, label = f'$Q^2=0$ $GeV^2$')
        axs[i*2, 1].axvline(x = qvcenter, color = 'brown', linestyle='dashdot', lw = 1)
                
        axs[i*2, 0].text(0.95, 0.95, f'$R_L \ (q=${qvcenter}$GeV$)', transform = axs[i*2, 0].transAxes, ha = 'right', va='top')
        axs[i*2, 1].text(0.95, 0.95, f'$R_T \ (q=${qvcenter}$GeV$)', transform = axs[i*2, 1].transAxes, ha = 'right', va='top')
        axs[i*2, 0].set_ylabel('$R_L \ (GeV^{-1})$')
        axs[i*2, 1].set_ylabel('$R_T \ (GeV^{-1})$')

        axs[i*2, 0].set_xlim(0.03, qvcenter * 1.05)
        axs[i*2, 1].set_xlim(0.03, qvcenter * 1.05)
        axs[i*2, 0].set_ylim(0, RL_plot_top[bin_index])
        axs[i*2, 1].set_ylim(0, max(RT_plot_top[bin_index], Photo_max))

        h, l = axs[i*2, 0].get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        h, l = axs[i*2, 1].get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        
        # Diff part
        SuSAV2_data = pd.read_csv(f'Data/Ca40_SuSAV2/qv/40Ca_BC_q_{int(qvcenter*1e3):04d}_new.dat', sep = '\s+', header = None)
        SuSAV2_data.columns = ['nu', 'RLQE', 'RTQE', 'RLMEC', 'RTMEC', 'RLinelastic', 'RTinelastic']
        SuSAV2_data['RLtotal'] = SuSAV2_data['RLQE'] + SuSAV2_data['RLMEC'] + SuSAV2_data['RLinelastic']
        SuSAV2_data['RTtotal'] = SuSAV2_data['RTQE'] + SuSAV2_data['RTMEC'] + SuSAV2_data['RTinelastic']
        SuSAV2_nu = SuSAV2_data['nu']
        SuSAV2_RL = SuSAV2_data['RLtotal']
        SuSAV2_RT = SuSAV2_data['RTtotal']
        
        before = pd.read_csv("Data/Ca40_RLRT_Numbers_Before.csv")
        after = pd.read_csv("Data/Ca40_RLRT_Numbers_After.csv")
        before = before[before['qvcenter'] == qvcenter]
        after = after[after['qvcenter'] == qvcenter]
        before['RL'] = before['RL']*1e3
        before['RLerr'] = before['RLerr']*1e3
        before['RT'] = before['RT']*1e3
        before['RTerr'] = before['RTerr']*1e3
        after['RL'] = after['RL']*1e3
        after['RLerr'] = after['RLerr']*1e3
        after['RT'] = after['RT']*1e3
        after['RTerr'] = after['RTerr']*1e3
        Our_nu = (before['nu'] + after['nu']) / 2
        Our_RL = (before['RL'] + after['RL']) / 2
        Our_RLerr = np.sqrt((before['RLerr']**2 + after['RLerr']**2) / 4 + (before['RL'] - after['RL'])**2 / 4)
        Our_RT = (before['RT'] + after['RT']) / 2
        Our_RTerr = np.sqrt((before['RTerr']**2 + after['RTerr']**2) / 4 + (before['RT'] - after['RT'])**2 / 4)
        Diff_RL = []
        Diff_RT = []
        for nu_val, our_rl, our_rlerr, our_rt, our_rterr in zip(
            Our_nu, Our_RL, Our_RLerr, Our_RT, Our_RTerr
        ):
            closest_idx = (SuSAV2_nu - nu_val).abs().idxmin()
            Diff_RL_this = our_rl - SuSAV2_RL.loc[closest_idx]
            Diff_RT_this = our_rt - SuSAV2_RT.loc[closest_idx]
            Diff_RL.append(Diff_RL_this)
            Diff_RT.append(Diff_RT_this)
            new_row = pd.Series({'qvcenter': qvcenter, 'Q2center': None, 'nu': nu_val, 'Diff_RL': Diff_RL_this, 'RLerr': our_rlerr, 'Diff_RT': Diff_RT_this, 'RTerr': our_rterr})
            with open(RLRT_number_file, 'a', newline='') as csvfile:
                        writer = csv.DictWriter(csvfile, fieldnames = new_row.index)
                        writer.writerow(new_row.to_dict())
        Diff_RL = np.array(Diff_RL)
        Diff_RT = np.array(Diff_RT)
        mask = (Our_nu > 0.03) & (Our_nu < qvcenter * 1.05)
        axs[i*2+1, 0].scatter(Our_nu[mask], Diff_RL[mask], color = 'red', label = "$R_L$, $R_T$ this - SuSAV2", **other_scatter_setting)
        axs[i*2+1, 0].errorbar(Our_nu[mask], Diff_RL[mask], yerr = Our_RLerr[mask], color = 'red', **errorbar_setting)
        axs[i*2+1, 1].scatter(Our_nu[mask], Diff_RT[mask], color ='red', **other_scatter_setting)
        axs[i*2+1, 1].errorbar(Our_nu[mask], Diff_RT[mask], yerr = Our_RTerr[mask], color = 'red', **errorbar_setting)
        
        W_peaks = np.array([0.93, 1.07, 1.23]) 
        W_colors = ['darkorange', 'turquoise', 'slateblue']
        W_locations = - mass_nucleon + np.sqrt(qvcenter**2 + W_peaks**2)
        for k in range(len(W_peaks)):
            location = W_locations[k]
            if location < qvcenter:
                axs[i*2+1, 0].axvline(x = location, color = W_colors[k], linestyle = 'dashdot', lw = 1, label = f'$W={W_peaks[k]}$ $GeV$')
                axs[i*2+1, 1].axvline(x = location, color = W_colors[k], linestyle = 'dashdot', lw = 1)
        
        axs[i*2+1, 0].axvline(x = qvcenter, color = 'brown', linestyle='dashdot', lw = 1, label = f'$Q^2=0$ $GeV^2$')
        axs[i*2+1, 1].axvline(x = qvcenter, color = 'brown', linestyle='dashdot', lw = 1)
        
        axs[i*2+1, 0].axhline(y = 0, color = 'gray', linestyle = '--', zorder = -10)
        axs[i*2+1, 1].axhline(y = 0, color = 'gray', linestyle = '--', zorder = -10)
        
        axs[i*2+1, 0].text(0.95, 0.95, f'$R_L Diff \ (q=${qvcenter}$GeV$)', transform = axs[i*2+1, 0].transAxes, ha = 'right', va = 'top')
        axs[i*2+1, 1].text(0.95, 0.95, f'$R_T Diff \ (q=${qvcenter}$GeV$)', transform = axs[i*2+1, 1].transAxes, ha = 'right', va = 'top')
        if i == len(bin_indices) - 1:
            axs[i*2+1, 0].set_xlabel('$\\nu \ (GeV)$')
            axs[i*2+1, 1].set_xlabel('$\\nu \ (GeV)$')
        axs[i*2+1, 0].set_ylabel('$R_L \ (GeV^{-1})$')
        axs[i*2+1, 1].set_ylabel('$R_T \ (GeV^{-1})$')
        axs[i*2+1, 0].set_xlim(0.03, qvcenter * 1.05)
        axs[i*2+1, 1].set_xlim(0.03, qvcenter * 1.05)
        axs[i*2+1, 0].set_ylim(- RL_plot_top[bin_index] / 2, RL_plot_top[bin_index] / 2)
        axs[i*2+1, 1].set_ylim(- RL_plot_top[bin_index] / 2, RL_plot_top[bin_index] / 2)

        target_ax = axs[i*2+1, 0]
        for spine in target_ax.spines.values():
            spine.set_edgecolor('gold')
            spine.set_linewidth(1.5)
        target_ax = axs[i*2+1, 1]
        for spine in target_ax.spines.values():
            spine.set_edgecolor('gold')
            spine.set_linewidth(1.5)
        
        h, l = axs[i*2+1, 0].get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        h, l = axs[i*2+1, 1].get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        
    unique_handles = []
    unique_labels = []
    for label in labels:
        if label not in unique_labels:
            unique_labels.append(label)
            unique_handles.append(handles[labels.index(label)])

    plt.subplots_adjust(bottom = 0.15)
    fig.legend(unique_handles, unique_labels, loc = 'lower center', ncol = 3, frameon = False)
    plt.show()
    
    return fig

def RLRT_plot_Q2bin(bin_indices = [0]):
    RL_plot_top = np.array([250, 250, 250, 250, 200, 
                            150, 100, 60, 50, 30, 
                            20, 10, 10, 3, 3, 
                            3, 3, 3])
    RT_plot_top = np.array([70, 70, 70, 70, 70, 
                            100, 100, 100, 80, 60, 
                            60, 30, 20, 10, 5, 
                            3, 3, 3])

    fig, axs = plt.subplots(nrows = len(bin_indices)*2, ncols = 2, figsize = (10, 8), dpi = 300)
    
    handles = []
    labels = []
    
    for i in range(len(bin_indices)):
        bin_index = bin_indices[i]
        Q2center = Q2centers[bin_index]
        
        # Normal part
        SuSAV2_data = pd.read_csv(f'Data/Ca40_SuSAV2/Q2/40Ca_BC_Q2_{int(Q2center*1e3):04d}_new.dat', sep = '\s+', header = None)
        SuSAV2_data.columns = ['nu', 'RLQE', 'RTQE', 'RLMEC', 'RTMEC', 'RLinelastic', 'RTinelastic']
        SuSAV2_data['RLtotal'] = SuSAV2_data['RLQE'] + SuSAV2_data['RLMEC'] + SuSAV2_data['RLinelastic']
        SuSAV2_data['RTtotal'] = SuSAV2_data['RTQE'] + SuSAV2_data['RTMEC'] + SuSAV2_data['RTinelastic']
        SuSAV2_data_slice = SuSAV2_data[SuSAV2_data['nu'] > 0].copy()
        line1, = axs[i*2, 0].plot(SuSAV2_data_slice['nu'], SuSAV2_data_slice['RLQE'], label = '$R_L$, $R_T$ (QE) SuSAv2', color = 'violet', linestyle = ':', lw = 3, zorder = -1)
        axs[i*2, 0].plot(SuSAV2_data_slice['nu'], SuSAV2_data_slice['RLtotal'], label = '$R_L$, $R_T$ (Total) SuSAv2', color = 'violet', linestyle = '-', lw = 2, zorder = -1)
        line2, = axs[i*2, 1].plot(SuSAV2_data_slice['nu'], SuSAV2_data_slice['RTQE'], color = 'violet', linestyle = ':', lw = 3, zorder = -1)
        axs[i*2, 1].plot(SuSAV2_data_slice['nu'], SuSAV2_data_slice['RTQE'] + SuSAV2_data_slice['RTMEC'], label = '$R_T$ (QE+2p2h) SuSAv2', color = 'violet', linestyle = '--', lw = 2, zorder = -1)
        axs[i*2, 1].plot(SuSAV2_data_slice['nu'], SuSAV2_data_slice['RTtotal'], color = 'violet', linestyle='-', lw = 2, zorder = -1)
        line1.set_path_effects([pe.Stroke(linewidth = 3.5, foreground = 'black'), pe.Normal()])
        line2.set_path_effects([pe.Stroke(linewidth = 3.5, foreground = 'black'), pe.Normal()])
        
        before = pd.read_csv("Data/Ca40_RLRT_Numbers_Before.csv")
        after = pd.read_csv("Data/Ca40_RLRT_Numbers_After.csv")
        before = before[before['Q2center'] == Q2center]
        after = after[after['Q2center'] == Q2center]
        before['RL'] = before['RL']*1e3
        before['RLerr'] = before['RLerr']*1e3
        before['RT'] = before['RT']*1e3
        before['RTerr'] = before['RTerr']*1e3
        after['RL'] = after['RL']*1e3
        after['RLerr'] = after['RLerr']*1e3
        after['RT'] = after['RT']*1e3
        after['RTerr'] = after['RTerr']*1e3
        axs[i*2, 0].scatter((before['nu'] + after['nu']) / 2, (before['RL'] + after['RL']) / 2, color = 'red', label = '$R_L$, $R_T$ this analysis', **our_scatter_setting)
        axs[i*2, 0].errorbar((before['nu'] + after['nu']) / 2, (before['RL'] + after['RL']) / 2, yerr = np.sqrt((before['RLerr']**2 + after['RLerr']**2) / 4 + (before['RL'] - after['RL'])**2 / 4), color = 'red', **errorbar_setting)
        axs[i*2, 1].scatter((before['nu'] + after['nu']) / 2, (before['RT'] + after['RT']) / 2, color = 'red', **our_scatter_setting)
        axs[i*2, 1].errorbar((before['nu'] + after['nu']) / 2, (before['RT'] + after['RT']) / 2, yerr = np.sqrt((before['RTerr']**2 + after['RTerr']**2) / 4 + (before['RT'] - after['RT'])**2 / 4), color = 'red', **errorbar_setting)
        
        if Q2center == 0.01:
            Photo_data = pd.read_csv('Data/Ca40_Photoproduction_Q2.csv')
            Photo_data['RT'] *= 1e3
            Photo_data['RTerr'] *= 1e3
            axs[i*2, 1].errorbar(Photo_data['nu'], Photo_data["RT"], yerr = Photo_data['RTerr'], label = 'Photo-production ($Q^2=0$ $GeV^2$)', markersize = 5, color = 'lime', marker = '^', markeredgecolor = 'black', markeredgewidth = 0.5, linewidth = 0.5, zorder = 1)
        
        W_peaks = np.array([0.93, 1.07, 1.23]) 
        W_colors = ['darkorange', 'turquoise', 'slateblue']
        W_locations = (W_peaks**2 + Q2center - mass_nucleon**2) / (2 * mass_nucleon)
        for k in range(len(W_peaks)):
            location = W_locations[k]
            axs[i*2, 0].axvline(x = location, color = W_colors[k], linestyle = 'dashdot', lw = 1, label = f'$W={W_peaks[k]}$ $GeV$')
            axs[i*2, 1].axvline(x = location, color = W_colors[k], linestyle = 'dashdot', lw = 1)
        
        axs[i*2 ,0].text(0.95, 0.95, f'$R_L \ (Q^2=${Q2center}$GeV^2$)', transform = axs[i*2, 0].transAxes, ha = 'right', va = 'top')
        axs[i*2, 1].text(0.95, 0.95, f'$R_T \ (Q^2=${Q2center}$GeV^2$)', transform = axs[i*2, 1].transAxes, ha = 'right', va = 'top')
        axs[i*2, 0].set_ylabel('$R_L \ (GeV^{-1})$')
        axs[i*2, 1].set_ylabel('$R_T \ (GeV^{-1})$')

        axs[i*2, 0].set_xlim(0.03, max(W_locations[2], after['nu'].max()) * 1.1)
        axs[i*2, 1].set_xlim(0.03, max(W_locations[2], after['nu'].max()) * 1.1)
        axs[i*2, 0].set_ylim(0, RL_plot_top[bin_index])
        axs[i*2, 1].set_ylim(0, RT_plot_top[bin_index])

        h, l = axs[i*2, 0].get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        h, l = axs[i*2, 1].get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        
        # Diff part
        SuSAV2_data = pd.read_csv(f'Data/Ca40_SuSAV2/Q2/40Ca_BC_Q2_{int(Q2center*1e3):04d}_new.dat', sep = '\s+', header = None)
        SuSAV2_data.columns = ['nu', 'RLQE', 'RTQE', 'RLMEC', 'RTMEC', 'RLinelastic', 'RTinelastic']
        SuSAV2_data['RLtotal'] = SuSAV2_data['RLQE'] + SuSAV2_data['RLMEC'] + SuSAV2_data['RLinelastic']
        SuSAV2_data['RTtotal'] = SuSAV2_data['RTQE'] + SuSAV2_data['RTMEC'] + SuSAV2_data['RTinelastic']
        SuSAV2_nu = SuSAV2_data['nu']
        SuSAV2_RL = SuSAV2_data['RLtotal']
        SuSAV2_RT = SuSAV2_data['RTtotal']
        
        before = pd.read_csv("Data/Ca40_RLRT_Numbers_Before.csv")
        after = pd.read_csv("Data/Ca40_RLRT_Numbers_After.csv")
        before = before[before['Q2center'] == Q2center]
        after = after[after['Q2center'] == Q2center]
        before['RL'] = before['RL']*1e3
        before['RLerr'] = before['RLerr']*1e3
        before['RT'] = before['RT']*1e3
        before['RTerr'] = before['RTerr']*1e3
        after['RL'] = after['RL']*1e3
        after['RLerr'] = after['RLerr']*1e3
        after['RT'] = after['RT']*1e3
        after['RTerr'] = after['RTerr']*1e3
        Our_nu = (before['nu'] + after['nu']) / 2
        Our_RL = (before['RL'] + after['RL']) / 2
        Our_RLerr = np.sqrt((before['RLerr']**2 + after['RLerr']**2) / 4 + (before['RL'] - after['RL'])**2 / 4)
        Our_RT = (before['RT'] + after['RT']) / 2
        Our_RTerr = np.sqrt((before['RTerr']**2 + after['RTerr']**2) / 4 + (before['RT'] - after['RT'])**2 / 4)
        Diff_RL = []
        Diff_RT = []
        for nu_val, our_rl, our_rlerr, our_rt, our_rterr in zip(
            Our_nu, Our_RL, Our_RLerr, Our_RT, Our_RTerr
        ):
            closest_idx = (SuSAV2_nu - nu_val).abs().idxmin()
            Diff_RL_this = our_rl - SuSAV2_RL.loc[closest_idx]
            Diff_RT_this = our_rt - SuSAV2_RT.loc[closest_idx]
            Diff_RL.append(Diff_RL_this)
            Diff_RT.append(Diff_RT_this)
            new_row = pd.Series({'qvcenter': None, 'Q2center': Q2center, 'nu': nu_val, 'Diff_RL': Diff_RL_this, 'RLerr': our_rlerr, 'Diff_RT': Diff_RT_this, 'RTerr': our_rterr})
            with open(RLRT_number_file, 'a', newline='') as csvfile:
                        writer = csv.DictWriter(csvfile, fieldnames = new_row.index)
                        writer.writerow(new_row.to_dict())
        Diff_RL = np.array(Diff_RL)
        Diff_RT = np.array(Diff_RT)
        mask = (Our_nu > 0.03) & (Our_nu < before['nu'].max() * 1.1)
        axs[i*2+1, 0].scatter(Our_nu[mask], Diff_RL[mask], color = 'red', label = "$R_L$, $R_T$ this - SuSAV2", **other_scatter_setting)
        axs[i*2+1, 0].errorbar(Our_nu[mask], Diff_RL[mask], yerr = Our_RLerr[mask], color = 'red', **errorbar_setting)
        axs[i*2+1, 1].scatter(Our_nu[mask], Diff_RT[mask], color ='red', **other_scatter_setting)
        axs[i*2+1, 1].errorbar(Our_nu[mask], Diff_RT[mask], yerr = Our_RTerr[mask], color = 'red', **errorbar_setting)

        W_peaks = np.array([0.93, 1.07, 1.23]) 
        W_colors = ['darkorange', 'turquoise', 'slateblue']
        W_locations = (W_peaks**2 + Q2center - mass_nucleon**2) / (2 * mass_nucleon)
        for k in range(len(W_peaks)):
            location = W_locations[k]
            axs[i*2+1, 0].axvline(x = location, color = W_colors[k], linestyle = 'dashdot', lw = 1, label = f'$W={W_peaks[k]}$ $GeV$')
            axs[i*2+1, 1].axvline(x = location, color = W_colors[k], linestyle = 'dashdot', lw = 1)

        axs[i*2+1, 0].axhline(y = 0, color='gray', linestyle='--', zorder = -10)
        axs[i*2+1, 1].axhline(y = 0, color='gray', linestyle='--', zorder = -10)
        
        axs[i*2+1, 0].text(0.95, 0.95, f'$R_L Diff \ (Q^2=${Q2center}$GeV^2$)', transform = axs[i*2+1, 0].transAxes, ha = 'right', va = 'top')
        axs[i*2+1, 1].text(0.95, 0.95, f'$R_T Diff \ (Q^2=${Q2center}$GeV^2$)', transform = axs[i*2+1, 1].transAxes, ha = 'right', va = 'top')
        if i == len(bin_indices) - 1:
            axs[i*2+1, 0].set_xlabel('$\\nu \ (GeV)$')
            axs[i*2+1, 1].set_xlabel('$\\nu \ (GeV)$')
        axs[i*2+1, 0].set_ylabel('$R_L \ (GeV^{-1})$')
        axs[i*2+1, 1].set_ylabel('$R_T \ (GeV^{-1})$')
        axs[i*2+1, 0].set_xlim(0.03, max(W_locations[2], after['nu'].max()) * 1.1)
        axs[i*2+1, 1].set_xlim(0.03, max(W_locations[2], after['nu'].max()) * 1.1)
        axs[i*2+1, 0].set_ylim(- RL_plot_top[bin_index] / 2, RL_plot_top[bin_index] / 2)
        axs[i*2+1, 1].set_ylim(- RL_plot_top[bin_index] / 2, RL_plot_top[bin_index] / 2)

        target_ax = axs[i*2+1, 0]
        for spine in target_ax.spines.values():
            spine.set_edgecolor('gold')
            spine.set_linewidth(1.5)
        target_ax = axs[i*2+1, 1]
        for spine in target_ax.spines.values():
            spine.set_edgecolor('gold')
            spine.set_linewidth(1.5)
        
        h, l = axs[i*2+1, 0].get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        h, l = axs[i*2+1, 1].get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        
    unique_handles = []
    unique_labels = []
    for label in labels:
        if label not in unique_labels:
            unique_labels.append(label)
            unique_handles.append(handles[labels.index(label)])

    plt.subplots_adjust(bottom = 0.15)
    fig.legend(unique_handles, unique_labels, loc = 'lower center', ncol = 3, frameon = False)
    plt.show()
    
    return fig
        

In [ ]:
plot_configs = [
    (RLRT_plot_qvbin, [0, 1, 2]),
    (RLRT_plot_qvbin, [3, 4, 5]),
    (RLRT_plot_qvbin, [6, 7, 8]),
    (RLRT_plot_qvbin, [9, 10, 11]),
    (RLRT_plot_qvbin, [12, 13, 14]),
    (RLRT_plot_qvbin, [15, 16, 17]),
    (RLRT_plot_Q2bin, [0, 1, 2]),
    (RLRT_plot_Q2bin, [3, 4, 5]),
    (RLRT_plot_Q2bin, [6, 7, 8]),
    (RLRT_plot_Q2bin, [9, 10, 11]),
    (RLRT_plot_Q2bin, [12, 13, 14]),
    (RLRT_plot_Q2bin, [15, 16, 17])
]

with PdfPages(f'Output/Ca40_SuSAV2_Diff.pdf') as RLRT_pdf:
    for plot_func, bins in plot_configs:
        fig = plot_func(bin_indices = bins)
        RLRT_pdf.savefig(fig)
        plt.close(fig)

print(datetime.now())